In [1]:
import os
import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {'sample_title': sample_title}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
# metadata_df = metadata_df.reset_index()

# Filter out rows where tissue column is 'Adjacent normal tissue'
metadata_df = metadata_df[metadata_df['tissue'] != 'Adjacent normal tissue']
print(metadata_df.shape)
metadata_df.head()


(63, 10)


,sample_title,tissue,age(0,genotype(1,tumor size (0,t stage (1,n stage (0,m stage (0,os time (months),OS status
samples,,,,,,,,,,
GSM5107730,TFE3-1,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,93.7,0
GSM5107731,TFE3-2,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 3,n0; 1:n1): 0,m0; 1:m1): 0,104.8,0
GSM5107732,TFE3-3,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 1,52.3333333333333,1
GSM5107733,TFE3-4,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,71.9,0
GSM5107734,TFE3-5,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 4,n0; 1:n1): 0,m0; 1:m1): 0,10,1


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"


df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None


url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)


df_combined = metadata_df.join(df_raw_counts, how='inner')
df_combined.reset_index()


data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_99511/779919605.py:12: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,age(0,genotype(1,tumor size (0,t stage (1,n stage (0,m stage (0,os time (months),OS status,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM5107730,TFE3-1,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,93.7,0,...,10557,44,43,30,13960,7480,61,5212,11,73
GSM5107731,TFE3-2,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 3,n0; 1:n1): 0,m0; 1:m1): 0,104.8,0,...,6054,11,8,9,6566,857,5,5133,0,24
GSM5107732,TFE3-3,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 1,52.3333333333333,1,...,99870,70,48,53,58547,11641,77,46534,30,342
GSM5107733,TFE3-4,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,71.9,0,...,16153,58,61,58,31831,13117,122,15677,9,45
GSM5107734,TFE3-5,Renal cell carcinoma,<18; 1: ≥18): 0,male; 2: female): 1,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 4,n0; 1:n1): 0,m0; 1:m1): 0,10,1,...,24443,33,19,21,20853,5552,87,13012,13,75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM5107788,TFE3-65,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,29,0,...,87637,195,173,154,55236,11290,184,44045,178,193
GSM5107789,TFE3-66,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 1,<median; 1: ≥median): 0,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 0,m0; 1:m1): 0,3,0,...,4999,34,45,40,4980,1507,18,2254,3,45
GSM5107790,TFE3-67,Renal cell carcinoma,<18; 1: ≥18): 1,male; 2: female): 2,<median; 1: ≥median): 1,t1; 2: t2; 3: t3; 4:t4): 1,n0; 1:n1): 1,m0; 1:m1): 0,2,0,...,235518,144,75,100,166651,37589,239,147982,176,397


In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)

desc_data['data_summary'] = {
    'samples': df_combined.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
